<a href="https://colab.research.google.com/github/Reben80/Data201/blob/main/DATA_201_Week_5_Part2_Feature_Selection_Linear_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Feature Selection for Linear Regression (Python)
This notebook supports a 75-minute class on **which predictors to include** in a linear regression model.

## Bridging for R users
R tools you already know:
- Fit: `lm(y ~ x1 + x2, data=df)`
- Compare: `AIC(m1, m2)`, `BIC(m1, m2)`
- Nested models / partial F-test: `anova(m_small, m_big)`
- VIF: `car::vif(m)`

Python equivalents used here:
- Fit: `statsmodels.OLS(y, X).fit()`
- Compare: `model.aic`, `model.bic`, `model.rsquared_adj`
- Partial F-test: `m_big.compare_f_test(m_small)`
- VIF: compute from regressions (function below)

**Key distinction:** Module 2 was *model form*. Today is *predictor inclusion*.


## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations

np.random.seed(12)


## 1. Teaching dataset (true predictors + redundant + noise + ID)
We simulate data so that:
- `x1`, `x2`, and `z` are truly predictive
- `x1_dup` is redundant with `x1` (multicollinearity)
- `noise1`, `noise2` are irrelevant
- `id` is an ID-like variable that should be excluded

In [2]:
n = 220
x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 1, n)
z = np.random.choice([0, 1], size=n)
x1_dup = x1 + np.random.normal(0, 0.10, n)
noise1 = np.random.normal(0, 1, n)
noise2 = np.random.normal(0, 1, n)
id_col = np.arange(1, n+1)
y = 4 + 2.5*x1 - 1.8*x2 + 1.2*z + np.random.normal(0, 1.2, n)

df = pd.DataFrame({
    'y': y,
    'x1': x1,
    'x2': x2,
    'z': z,
    'x1_dup': x1_dup,
    'noise1': noise1,
    'noise2': noise2,
    'id': id_col
})
df.head()

,y,x1,x2,z,x1_dup,noise1,noise2,id
0,6.758177,0.472986,0.083527,0,0.605410,-0.767692,-0.356720,1
1,5.354773,-0.681426,-1.658425,0,-0.782725,-0.471337,-0.837435,2
2,6.140532,0.242439,0.380114,1,0.276946,0.662876,-1.334925,3
3,2.523129,-1.700736,-0.956673,1,-1.667830,-0.761782,0.121829,4
4,4.774294,0.753143,1.269859,0,0.785762,0.537063,1.352733,5


### R equivalent
```r
# df <- data.frame(y, x1, x2, z, x1_dup, noise1, noise2, id)
```

## 2. Remove obvious junk (IDs / leakage) and define candidates

In [3]:
candidates = ['x1','x2','z','x1_dup','noise1','noise2']
candidates

['x1', 'x2', 'z', 'x1_dup', 'noise1', 'noise2']

## 3. Redundancy check: correlation matrix

In [4]:
df[candidates].corr()

,x1,x2,z,x1_dup,noise1,noise2
x1,1.000000,0.146700,0.007252,0.995194,-0.036348,-0.119916
x2,0.146700,1.000000,0.055565,0.145503,-0.000857,0.000330
z,0.007252,0.055565,1.000000,0.009125,-0.114217,-0.009560
x1_dup,0.995194,0.145503,0.009125,1.000000,-0.038201,-0.119712
noise1,-0.036348,-0.000857,-0.114217,-0.038201,1.000000,0.080512
noise2,-0.119916,0.000330,-0.009560,-0.119712,0.080512,1.000000


### R equivalent
```r
# cor(df[, candidates])
```

## 4. Multicollinearity check: VIF
VIF is based on how well each predictor can be predicted from the others.
Higher VIF ⇒ more multicollinearity.

In [5]:
def vif_table(dataframe, features):
    out = []
    for f in features:
        X = dataframe[[c for c in features if c != f]]
        X = sm.add_constant(X)
        y = dataframe[f]
        r2 = sm.OLS(y, X).fit().rsquared
        vif = 1.0 / (1.0 - r2) if r2 < 0.999999 else np.inf
        out.append((f, r2, vif))
    return pd.DataFrame(out, columns=['feature','R2_from_others','VIF']).sort_values('VIF', ascending=False)

vif_table(df, candidates)

,feature,R2_from_others,VIF
0,x1,0.990422,104.404626
3,x1_dup,0.990419,104.371042
1,x2,0.024957,1.025595
5,noise2,0.020508,1.020938
4,noise1,0.020457,1.020885
2,z,0.016452,1.016727


### R equivalent
```r
# library(car)
# m <- lm(y ~ x1 + x2 + z + x1_dup + noise1 + noise2, data=df)
# vif(m)
```

**Decision point:** Drop one of the redundant predictors (`x1` vs `x1_dup`).
We keep `x1` and drop `x1_dup` for interpretability and stability.

In [6]:
candidates_pruned = ['x1','x2','z','noise1','noise2']
vif_table(df, candidates_pruned)

,feature,R2_from_others,VIF
0,x1,0.036649,1.038043
1,x2,0.024923,1.025560
4,noise2,0.020504,1.020933
3,noise1,0.020128,1.020542
2,z,0.016149,1.016414


## 5. Baseline model and key criteria: adjusted R², AIC, BIC

In [7]:
X_base = sm.add_constant(df[candidates_pruned])
m_base = sm.OLS(df['y'], X_base).fit()
pd.Series({
    'R2': m_base.rsquared,
    'Adj_R2': m_base.rsquared_adj,
    'AIC': m_base.aic,
    'BIC': m_base.bic
}).round(4)

R2          0.8573
Adj_R2      0.8540
AIC       696.3458
BIC       716.7076
dtype: float64

### R equivalent
```r
# m_base <- lm(y ~ x1 + x2 + z + noise1 + noise2, data=df)
# summary(m_base)$adj.r.squared
# AIC(m_base); BIC(m_base)
```

## 6. Subset comparison (teaching demo)
For teaching, we do best-subset search over 5 predictors and compare by BIC/AIC/Adj R².
In real projects, you would compare a **small set of motivated candidates** rather than brute force.

In [8]:
def fit_subset(features):
    X = sm.add_constant(df[list(features)])
    m = sm.OLS(df['y'], X).fit()
    return {
        'features': tuple(features),
        'k': len(features),
        'Adj_R2': m.rsquared_adj,
        'AIC': m.aic,
        'BIC': m.bic,
        'model': m
    }

results = []
for k in range(1, len(candidates_pruned)+1):
    for feats in combinations(candidates_pruned, k):
        results.append(fit_subset(feats))

res_df = pd.DataFrame([{k:v for k,v in r.items() if k!='model'} for r in results])
res_df.sort_values('BIC').head(10)

,features,k,Adj_R2,AIC,BIC
15,"(x1, x2, z)",3,0.854609,693.424838,706.999348
25,"(x1, x2, z, noise1)",4,0.854590,694.432714,711.400852
26,"(x1, x2, z, noise2)",4,0.854023,695.288650,712.256787
30,"(x1, x2, z, noise1, noise2)",5,0.853968,696.345785,716.707550
5,"(x1, x2)",2,0.831494,724.900405,735.081287
17,"(x1, x2, noise2)",3,0.830777,726.817759,740.392269
16,"(x1, x2, noise1)",3,0.830773,726.823011,740.397521
27,"(x1, x2, noise1, noise2)",4,0.830041,728.751735,745.719872
6,"(x1, z)",2,0.607410,910.975157,921.156040
0,"(x1,)",1,0.593159,917.830666,924.617921


Best model by each criterion:

In [ ]:
best = pd.concat([
    res_df.sort_values('BIC').head(1).assign(criterion='Best BIC'),
    res_df.sort_values('AIC').head(1).assign(criterion='Best AIC'),
    res_df.sort_values('Adj_R2', ascending=False).head(1).assign(criterion='Best Adj R2')
])[['criterion','features','k','Adj_R2','AIC','BIC']]
best

### R equivalents
```r
# m_full <- lm(y ~ x1 + x2 + z + noise1 + noise2, data=df)
# step(m_full, direction='both')                  # AIC stepwise
# step(m_full, direction='both', k=log(nrow(df))) # BIC-like stepwise
```

## 7. Partial F-test logic (nested models)
Compare reduced vs full:
- Reduced: `y ~ x1 + x2 + z`
- Full: `y ~ x1 + x2 + z + noise1 + noise2`
A large p-value suggests the added terms do not earn their complexity.

In [9]:
m_red = sm.OLS(df['y'], sm.add_constant(df[['x1','x2','z']])).fit()
m_full = m_base
F_stat, p_val, df_diff = m_full.compare_f_test(m_red)
pd.Series({'F': F_stat, 'p_value': p_val, 'df_diff': df_diff})

F          0.526101
p_value    0.591666
df_diff    2.000000
dtype: float64

### R equivalent
```r
# m_red <- lm(y ~ x1 + x2 + z, data=df)
# m_full <- lm(y ~ x1 + x2 + z + noise1 + noise2, data=df)
# anova(m_red, m_full)
```

## 8. Final model (example)
Combine evidence (AIC/BIC/Adj R² + multicollinearity + domain reasoning) to choose a final predictor set.

In [10]:
m_final = m_red
pd.Series({'Adj_R2': m_final.rsquared_adj, 'AIC': m_final.aic, 'BIC': m_final.bic}).round(4)

Adj_R2      0.8546
AIC       693.4248
BIC       706.9993
dtype: float64

## Exit ticket (concept check)
1) Why is R² alone a poor criterion for feature selection?
2) When might BIC select a smaller model than AIC?
3) What does a high VIF indicate and what is a reasonable response?
4) Give one domain-driven reason to keep a predictor even if it is not statistically strong.
